# Purpose
This notebook gets adjacency matrices from datasets of empirical connectomes

In [ ]:
import pickle
import pandas as pd
import xml.etree.ElementTree as ET
import csv
import sys
sys.path.append('..')
from scripts import clean_data

In [ ]:
A_real = {}

## C. elegans

In [ ]:
# from xml to csv of edge list
tree = ET.parse("adjacency_adult.xml")
root = tree.getroot()

with open("./Output/ce_output.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["section", "cell1", "cell2", "index1", "index2", "adjacency"])

    for layer in root.findall("layer"):
        section_name = layer.get("name")
        for area in layer.findall("area"):
            row = [section_name, area.find("cell1").text,
                   area.find("cell2").text, area.find("index1").text,
                   area.find("index2").text,area.find("adjacency").text,]
            writer.writerow(row)
print("saved")

In [ ]:
# from edge list to adjacency matrix
df = pd.read_csv('./Output/ce_output.csv')
columns = df.columns.tolist()
result = (df.groupby(["cell1", "cell2"], as_index=False).agg({"adjacency": "sum"}))
arr = clean_data.get_adj(result)
arr = arr.T
arr = arr*4.5e-4
arr.shape

In [ ]:
A_real['C.elegans'] = arr.copy()

## Mouse retina

In [ ]:
# from edge list to adjacency matrix
df = pd.read_csv('Mouse_retina.csv')
edge_ls = df.groupby(['pre', 'post'], as_index=False).agg({"weight": "sum"})
arr = clean_data.get_adj(edge_ls)

In [ ]:
A_real['Mouse retina'] = arr.copy()

In [ ]:
f = open('./Output/Experimental Adj.pckl', 'wb')
pickle.dump(A_real, f)
f.close()